### NLBSE'24 SetFit Implementation for Valid Bug Classification

# Data Rearranging Script

In [ ]:
from datasets import load_dataset, concatenate_datasets

ds = load_dataset(
    "csv",
    data_files={
        "train": "data/issues_train.csv",
        "test": "data/issues_test.csv",
    }
)

combined = concatenate_datasets([ds["train"], ds["test"]])

In [ ]:
def map_valid_invalid(example):
    return {
        "label": 1 if example["label"] == "bug" else 0
    }

In [ ]:
from datasets import ClassLabel

combined = combined.map(map_valid_invalid)
combined = combined.cast_column("label", ClassLabel(names=['invalid', 'valid']))

Casting the dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [ ]:
new_ds = combined.train_test_split(
    train_size=2500,
    test_size=500,
    stratify_by_column="label",
    seed=42,
)

In [ ]:
new_ds

DatasetDict({
    train: Dataset({
        features: ['repo', 'created_at', 'label', 'title', 'body'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['repo', 'created_at', 'label', 'title', 'body'],
        num_rows: 500
    })
})

In [ ]:
new_ds["train"].to_csv("data/new_train.csv", index=False)
new_ds["test"].to_csv("data/new_test.csv", index=False)

print("Train and test datasets saved to data/new_train.csv and data/new_test.csv")

Creating CSV from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Train and test datasets saved to data/new_train.csv and data/new_test.csv


# Original Script

In [ ]:
!pip install -q setfit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00


In [ ]:
BASE_MODEL = "sentence-transformers/all-mpnet-base-v2"
RANDOM_SEED = 42
OUTPUT_PATH = 'output/setfit'

!mkdir -p $OUTPUT_PATH

In [ ]:
DATA_PATH = 'data'

!mkdir -p $DATA_PATH

In [ ]:
from datasets import Dataset

ds = Dataset.from_csv({ "train": "data/issue_train.csv", "test": "data/issue_test.csv" })
ds

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['repo', 'created_at', 'label', 'title', 'body'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['repo', 'created_at', 'label', 'title', 'body'],
        num_rows: 500
    })
})

In [ ]:
repos = ds["train"].unique("repo")
print(repos)

['opencv/opencv', 'microsoft/vscode', 'bitcoin/bitcoin', 'facebook/react', 'tensorflow/tensorflow']


In [ ]:
ds["train"].to_pandas().groupby(["repo", "label"]).size().unstack(fill_value=0)

label,0,1
repo,,
bitcoin/bitcoin,338,170
facebook/react,336,161
microsoft/vscode,325,168
opencv/opencv,341,173
tensorflow/tensorflow,327,161


In [ ]:
import re

def process_dataset(example):

    # concatenate title and body
    text = (example['title'] or "") + " " + (example['body'] or "")

    example['text'] = text
    return example

In [ ]:
ds = ds.shuffle(seed=RANDOM_SEED)
ds = ds.map(process_dataset)
ds = ds.select_columns(['repo', 'label', 'text'])
ds

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['repo', 'label', 'text'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['repo', 'label', 'text'],
        num_rows: 500
    })
})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_SAVE_DIR = "/content/drive/MyDrive/CS559/project/mpnet_models"

In [14]:
import wandb
from datetime import datetime
from setfit import SetFitModel, Trainer, TrainingArguments
import os

group = datetime.utcnow().replace(microsecond=0).isoformat()

references = {}
predictions = {}

wandb.init(
    project="NLBSE'24 Issue Report Classification - SetFit",
    group=group,
    name="bug_validity_classification",
)

model = SetFitModel.from_pretrained(BASE_MODEL)

args = TrainingArguments(
    output_dir=OUTPUT_PATH,
    save_strategy="no",
    report_to="wandb",
    logging_steps=1,
    seed=RANDOM_SEED,
    batch_size=(32, 2),
    num_epochs=1,
    num_iterations=5,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
)

trainer.train()

/tmp/ipython-input-3090106493.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  group = datetime.utcnow().replace(microsecond=0).isoformat()


train/embedding_loss,▇▇█▆▇▅▅▆▄▅▃▂▂▄▂▃▁▁▂▂▂▂▃▂▁▁▂▂▁▁▁▂▁▂▂▁▁▁▁▁
train/epoch,▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇█
train/global_step,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇█████
train/grad_norm,▃▂▂▂▂▃▃▃▄▄▃▃▂▄▃▂▃▄▁▁▁▁▃▁▂▁▃▄▂▃▁▁▃▁█▁▄▁▂▁
train/learning_rate,▁▁▂▂▂▃▃▄▅▅▆▇██████████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▆
train/embedding_loss,0.0002
train/epoch,0.30208
train/global_step,944
train/grad_norm,0.0313
train/learning_rate,2e-05


/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:263: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
***** Running training *****
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
  Num unique pairs = 25000
  Batch size = 32
  Num epochs = 1


Step,Training Loss
1,0.429200
2,0.443800
3,0.339200
4,0.377400
5,0.416700
6,0.328100
7,0.430700
8,0.319500
9,0.294400
10,0.339700


Step,Training Loss
1,0.429200
2,0.443800
3,0.339200
4,0.377400
5,0.416700
6,0.328100
7,0.430700
8,0.319500
9,0.294400
10,0.339700


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
save_path = os.path.join(BASE_SAVE_DIR, "best_model")

model.save_pretrained(save_path)

In [16]:
test_set = ds["test"]

references = list(test_set['label'])

predictions = list(model.predict(test_set['text'], batch_size=8, show_progress_bar=True))

wandb.finish()

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


train/embedding_loss,███▆▆▆▆▅▅▄▅▄▄▂▂▂▁▂▁▃▃▂▃▃▂▂▁▂▂▁▂▁▂▁▁▁▁▁▂▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇████
train/global_step,▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▃▃▄▃▂▅▄▅▆▇▄▇▂▃▅▄▄▅▅▇▁█▄▇█▁▅▄▁▆▃▃▂▁▂▂▁▂▁▁
train/learning_rate,██████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▁▁
total_flos,0
train/embedding_loss,0.001
train/epoch,1
train/global_step,782
train/grad_norm,0.05478
train/learning_rate,0.0


In [18]:
from sklearn.metrics import classification_report
from numpy import mean

metrics = ['precision', 'recall', 'f1-score']

results = classification_report(references, predictions, digits=4, output_dict=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
results

{'0': {'precision': 0.9069767441860465,
  'recall': 0.9369369369369369,
  'f1-score': 0.9217134416543574,
  'support': 333.0},
 '1': {'precision': 0.8653846153846154,
  'recall': 0.8083832335329342,
  'f1-score': 0.8359133126934984,
  'support': 167.0},
 'accuracy': 0.894,
 'macro avg': {'precision': 0.8861806797853309,
  'recall': 0.8726600852349355,
  'f1-score': 0.8788133771739279,
  'support': 500.0},
 'weighted avg': {'precision': 0.8930849731663685,
  'recall': 0.894,
  'f1-score': 0.8930561985814304,
  'support': 500.0}}

In [22]:
import json
import os

output_file_name = 'results.json'

labels = ['0','1']

with open(os.path.join(OUTPUT_PATH, output_file_name), 'w') as fp:
    json.dump(results, fp, indent=2)

print(f"Repository{' '*15}Label     Precision  Recall     F1")
for repo in ['overall']:
  print("-"*63)
  for label in labels:
    out = f"{repo:<25}{label:<10}"
    for metric in metrics:
      out += f"{results[label][metric]:<10.4f} "
    print(out)

Repository               Label     Precision  Recall     F1
---------------------------------------------------------------
overall                  0         0.9070     0.9369     0.9217     
overall                  1         0.8654     0.8084     0.8359     
